# 03 - Embedding Generation & FAISS Index Creation

**Bio-Heritage AI: RAG-Based Sri Lankan Herb Knowledge Assistant**

Step 3 of the methodology:
1. Load the herb chunks from Day 1 (`chunks/herb_chunks.json`).
2. Embed each `chunk_text` using the `all-MiniLM-L6-v2` sentence-transformer.
3. Build a FAISS index for fast semantic search.
4. Save `indexes/sl_herb_faiss.index` and `indexes/herb_metadata.pkl`.
5. Quick test: search the index with an example query.

> First run downloads the model (~90 MB). Needs internet once, then works offline.

In [1]:
import json
import os
import pickle
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Paths (works whether you run from project root or notebooks/)
CHUNKS_PATH = "../chunks/herb_chunks.json"
if not os.path.exists(CHUNKS_PATH):
    CHUNKS_PATH = "chunks/herb_chunks.json"

INDEX_DIR = os.path.join(os.path.dirname(CHUNKS_PATH), "..", "indexes")
INDEX_DIR = os.path.normpath(INDEX_DIR)
os.makedirs(INDEX_DIR, exist_ok=True)
INDEX_PATH = os.path.join(INDEX_DIR, "sl_herb_faiss.index")
META_PATH = os.path.join(INDEX_DIR, "herb_metadata.pkl")
print("Chunks:", os.path.abspath(CHUNKS_PATH))
print("Index will be saved to:", os.path.abspath(INDEX_PATH))

c:\Users\THANUJA\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chunks: c:\Users\THANUJA\OneDrive\Documents\Bio-Heritage-Ai\chunks\herb_chunks.json
Index will be saved to: c:\Users\THANUJA\OneDrive\Documents\Bio-Heritage-Ai\indexes\sl_herb_faiss.index


In [2]:
# Load the chunks
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

texts = [c["chunk_text"] for c in chunks]
print("Loaded", len(chunks), "chunks.")
print("Example text preview:\n", texts[0][:200], "...")

Loaded 1550 chunks.
Example text preview:
 Herb (Sinhala): Gotukola
Herb (English): Asiatic pennywort
Latin name: Centella asiatica
Family: Apiaceae
Used for (treatment): memory improvement; wound healing; skin diseases
Parts used in treatment ...


In [3]:
# Load the sentence-transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded. Embedding dimension:", model.get_sentence_embedding_dimension())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5749.22it/s]


Model loaded. Embedding dimension: 384


C:\Users\THANUJA\AppData\Local\Temp\ipykernel_21860\4022064528.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Model loaded. Embedding dimension:", model.get_sentence_embedding_dimension())


In [4]:
# Generate embeddings for all chunks.
# normalize_embeddings=True lets us use inner-product = cosine similarity.
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

print("Embeddings shape:", embeddings.shape)

Batches: 100%|██████████| 25/25 [00:52<00:00,  2.08s/it]

Embeddings shape: (1550, 384)


In [5]:
# Build the FAISS index. IndexFlatIP = inner product (cosine, since vectors are normalized).
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print("FAISS index built. Total vectors:", index.ntotal)

# Save the index
faiss.write_index(index, INDEX_PATH)

# Save the metadata (same order as the vectors) so we can look records up after a search
with open(META_PATH, "wb") as f:
    pickle.dump(chunks, f)

print("Saved index ->", INDEX_PATH)
print("Saved metadata ->", META_PATH)

FAISS index built. Total vectors: 1550
Saved index -> ..\indexes\sl_herb_faiss.index
Saved metadata -> ..\indexes\herb_metadata.pkl


In [6]:
# --- Quick test: semantic search ---
def search(query, k=5):
    q_emb = model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(q_emb, k)
    results = []
    for score, i in zip(scores[0], idxs[0]):
        c = chunks[i]
        results.append((round(float(score), 3), c["herb_name_english"], c["herb_name_sinhala"], c["treatment_for"]))
    return results

print("Query: 'herbs for diabetes'\n")
for score, en, si, treat in search("herbs for diabetes"):
    print(f"  {score}  {en} ({si}) -> {treat}")

print("\nQuery: 'what is good for memory improvement'\n")
for score, en, si, treat in search("what is good for memory improvement"):
    print(f"  {score}  {en} ({si}) -> {treat}")

print("\nDay 2 (FAISS) done. Next: 02_intent_classifier_training.ipynb")

Query: 'herbs for diabetes'

  0.552  village creeper (Nisatana) -> urinary stones; chest congestion
  0.534  mountain medick (Rawitana) -> lactation support; dysentery; eye health; kidney health
  0.53  village tonic-vine (Daguessa) -> migraine; diabetes
  0.515  sacred sorrel (Dimaliya) -> immunity support; loss of appetite; intestinal worms; diabetes
  0.509  mountain ginger-lily (Nitakanda) -> kidney health; diarrhoea; cough; diabetes

Query: 'what is good for memory improvement'

  0.221  highland betony (Kamamula) -> memory improvement; inflammation
  0.21  forest balmwort (Pinamula) -> memory improvement; headache
  0.208  forest spurge (Kenariya) -> memory improvement; burns
  0.206  river medick (Mumamula) -> bronchitis; memory improvement
  0.206  coastal spurge (Muluaru) -> memory improvement; oedema

Day 2 (FAISS) done. Next: 02_intent_classifier_training.ipynb
